# DeSCOPE knockout-age screening

Paper: Fig. 6b–d, downstream analysis only. Inputs: original donor-by-knockout `perturbed_ages.csv` from the DeSCOPE inference and PeriClock prediction stage. Environment: Python, NumPy, pandas, SciPy, statsmodels, Matplotlib and GSEApy. Outputs: per-gene statistics, selected genes, enrichment tables and RNA-processing figures under `OUTPUT_DIR/results` and `OUTPUT_DIR/fig`. Execution starts from the external age table; there is no dependency on previous tutorial variables. **Upstream DeSCOPE inference, baseline generation and full screening execution remain unverified.** Missing upstream inference code/resources are not replaced by a placeholder workflow.

Original source indices below are **zero-based**. Original cell hashes are retained in cell metadata. Outputs were cleared; inclusion does not imply successful execution.

## Required resources and configuration

Original `perturbed_ages.csv`, matching GO BP 2023/Reactome 2022 GMTs and licensed Arial fonts; [processed inputs](../RESOURCE_DOWNLOAD.md#4-training-feature-matrices) and [other references](../RESOURCE_DOWNLOAD.md#8-other-reference-databases). Upstream DeSCOPE inference is not provided by this downstream notebook.

Download locations, sizes, SHA256, versions, permissions and publication status are maintained once in [RESOURCE_DOWNLOAD.md](../RESOURCE_DOWNLOAD.md). Individual GEO/GSM sources and sample choices are in the paper’s Supplementary Table 1. Clock models are uploaded to [Hugging Face Models](https://huggingface.co/LeyiZhang1/PeriClock); [Dataset uploads](https://huggingface.co/datasets/LeyiZhang1/PeriClock-data) are in progress. Both repositories are currently private. Consult the resource guide for per-file availability; donor-level public redistribution remains subject to review.

Set `PERICLOCK_RESOURCE_DIR` for external data and `PERICLOCK_MODEL_DIR` for models, or override individual file paths in the next cell. Keep outputs separate from inputs. Start a fresh kernel and run in order.


In [ ]:
from pathlib import Path
import os
# Start Jupyter from within this GitHub checkout.
PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                     if (p / "periclock").is_dir() and (p / "pyproject.toml").is_file()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("Start the notebook kernel from within the PeriClock code checkout")
RESOURCE_DIR = Path(os.environ.get("PERICLOCK_RESOURCE_DIR", "~/periclock-resources")).expanduser().resolve()
MODEL_DIR = Path(os.environ.get("PERICLOCK_MODEL_DIR", str(RESOURCE_DIR / "models"))).expanduser().resolve()
# Each individual file-path environment variable still overrides these defaults.
KO_AGES = Path(os.environ.get("PERICLOCK_KO_AGES", str(RESOURCE_DIR / "datasets/perturbed_ages.csv"))).expanduser().resolve()
GMT_DIR = Path(os.environ.get("PERICLOCK_GMT_DIR", str(RESOURCE_DIR / "external/enrichr"))).expanduser().resolve()
FONT_DIR = Path(os.environ.get("PERICLOCK_FONT_DIR", "/path/to/Arial")).expanduser().resolve()
OUTPUT_DIR = Path(os.environ.get("PERICLOCK_TUTORIAL_OUTPUT", str(PROJECT_ROOT / "outputs/08_descope_screening"))).expanduser().resolve()
# Input files remain read-only; all relative output paths below use this directory.
for input_path in [KO_AGES]:
    if not input_path.is_file():
        raise FileNotFoundError(f"Set the input path in this cell: {input_path}")
if not FONT_DIR.is_dir():
    raise FileNotFoundError("Set PERICLOCK_FONT_DIR to a directory containing licensed Arial fonts")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(OUTPUT_DIR)

## Per-gene knockout statistics and screening figure (Fig. 6b)

Source: cells 7 and 10. Required CSV columns: `sample`, `gene`, `delta_age`; the original screening uses `delta_age` as already calculated upstream. It checks duplicate sample–gene pairs, uses at least 30 donors, rounds only test values to six decimals, applies two-sided Wilcoxon and BH, and retains the original five-year / 70% strong-effect thresholds. The provided CSV defines `delta_age = ko_pred_age - baseline_pred_age`, projected from upstream `delta_age_vs_baseline`. It does not contain `null_pred_age`; upstream KO-minus-null is a distinct column. The upstream worker uses zero-embedding DeSCOPE output to calculate expression changes before clock prediction. Tutorial 08 tests the stored CSV differences without recomputing or replacing their baseline. This notebook cannot regenerate upstream perturbation inference. No DeSCOPE checkpoint, embedding or patient-level CSV is included.

In [ ]:
import os
import numpy as np
import pandas as pd
from scipy.stats import wilcoxon
from statsmodels.stats.multitest import multipletests

# ====
# ========================================================
# 1. 参数与数据检查
# ============================================================
outdir = "results/KO_screening"
os.makedirs(outdir, exist_ok=True)

FDR_CUTOFF = 0.05
MIN_SAMPLES = 30
MIN_EFFECT = 5.0
MIN_CONSISTENCY = 0.70

# 直接使用前面已经简化的df
df = pd.read_csv(str(KO_AGES))

print(f"总记录数: {len(df):,}")
print(f"基因数: {df['gene'].nunique():,}")
print(f"样本数: {df['sample'].nunique():,}")

# 检查重复sample-gene组合
n_dup = df.duplicated(["sample", "gene"]).sum()
print(f"重复sample-gene组合: {n_dup}")
if n_dup > 0:
    raise ValueError("存在重复sample-gene记录，请检查后再进行统计检验。")

# 过滤缺失值和非有限值
df = df.dropna(subset=["sample", "gene", "delta_age"])
df = df[np.isfinite(df["delta_age"])].copy()

# ============================================================
# 2. 逐基因计算扰动效应与Wilcoxon检验
# ============================================================
results = []

for gene, sub in df.groupby("gene", sort=False):
    x = sub["delta_age"].to_numpy(dtype=float)
    n = len(x)

    # 仅在统计检验中保留6位小数，减少浮点精度影响
    x_test = np.round(x, 6)

    if n < MIN_SAMPLES:
        pvalue = np.nan
    elif np.all(x_test == 0):
        pvalue = 1.0
    else:
        pvalue = wilcoxon(x_test, alternative="two-sided",
                         zero_method="wilcox", method="auto").pvalue

    results.append({
        "gene": gene,
        "n_samples": n,
        "mean_delta": np.mean(x),
        "median_delta": np.median(x),
        "sd_delta": np.std(x, ddof=1) if n > 1 else np.nan,
        "q25_delta": np.percentile(x, 25),
        "q75_delta": np.percentile(x, 75),
        "frac_decrease": np.mean(x < 0),
        "frac_increase": np.mean(x > 0),
        "frac_decrease_1yr": np.mean(x <= -5),
        "frac_increase_1yr": np.mean(x >= 5),
        "pvalue": pvalue
    })

gene_stats = pd.DataFrame(results)

# ============================================================
# 3. BH多重检验校正
# ============================================================
gene_stats["fdr"] = np.nan
valid = gene_stats["pvalue"].notna()

gene_stats.loc[valid, "fdr"] = multipletests(
    gene_stats.loc[valid, "pvalue"].values, method="fdr_bh"
)[1]

gene_stats["significant"] = gene_stats["fdr"] < FDR_CUTOFF

# 根据Median ΔAge确定方向
gene_stats["direction"] = np.select(
    [gene_stats["median_delta"] < 0, gene_stats["median_delta"] > 0],
    ["Decrease", "Increase"],
    default="No change"
)

gene_stats["consistency"] = np.where(
    gene_stats["median_delta"] < 0,
    gene_stats["frac_decrease"],
    np.where(gene_stats["median_delta"] > 0, gene_stats["frac_increase"], 0)
)

# ============================================================
# 4. 第一层：所有统计显著基因
# ============================================================
sig_all = gene_stats[gene_stats["significant"]].copy()

sig_down = sig_all[sig_all["median_delta"] < 0].sort_values("median_delta")
sig_up = sig_all[sig_all["median_delta"] > 0].sort_values("median_delta", ascending=False)

# ============================================================
# 5. 第二层：具有较强且稳定效应的基因
# ============================================================
strong = sig_all[
    (sig_all["median_delta"].abs() >= MIN_EFFECT) &
    (sig_all["consistency"] >= MIN_CONSISTENCY)
].copy()

strong_down = strong[strong["median_delta"] < 0].sort_values("median_delta")
strong_up = strong[strong["median_delta"] > 0].sort_values("median_delta", ascending=False)

# ============================================================
# 6. 汇总与保存
# ============================================================
print("\n========== 全基因KO筛选结果 ==========")
print(f"成功检验的基因: {valid.sum():,}")
print(f"统计显著基因(FDR < 0.05): {len(sig_all):,}")
print(f"  显著降低年龄: {len(sig_down):,}")
print(f"  显著升高年龄: {len(sig_up):,}")
print(f"较强且稳定的候选基因: {len(strong):,}")
print(f"  降低年龄: {len(strong_down):,}")
print(f"  升高年龄: {len(strong_up):,}")

gene_stats.sort_values("fdr").to_csv(f"{outdir}/all_gene_statistics.csv", index=False)
sig_all.to_csv(f"{outdir}/significant_genes.csv", index=False)
sig_down.to_csv(f"{outdir}/significant_decrease_genes.csv", index=False)
sig_up.to_csv(f"{outdir}/significant_increase_genes.csv", index=False)
strong_down.to_csv(f"{outdir}/strong_decrease_genes.csv", index=False)
strong_up.to_csv(f"{outdir}/strong_increase_genes.csv", index=False)

# ============================================================
# 7. 展示Top基因
# ============================================================
show_cols = ["gene", "n_samples", "mean_delta", "median_delta",
             "consistency", "frac_decrease", "frac_increase", "pvalue", "fdr"]

print("\nTop 20 显著降低预测年龄的基因:")
display(sig_down[show_cols].head(20).round(4))

print("\nTop 20 显著升高预测年龄的基因:")
display(sig_up[show_cols].head(20).round(4))

print("\nTop 20 强效且稳定降低预测年龄的基因:")
display(strong_down[show_cols].head(20).round(4))

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.font_manager as fm
from matplotlib.lines import Line2D

# ============================================================
# 1. 字体与全局参数
# ============================================================
font_dir = str(FONT_DIR)
if os.path.isdir(font_dir):
    for f in os.listdir(font_dir):
        if f.lower().endswith((".ttf", ".otf")):
            fm.fontManager.addfont(os.path.join(font_dir, f))

mpl.rcParams.update({
    "font.family": "Arial",
    "font.size": 12,
    "axes.linewidth": 1.2,
    "pdf.fonttype": 42,
    "ps.fonttype": 42
})

outdir = str(OUTPUT_DIR / "fig")
os.makedirs(outdir, exist_ok=True)

# ============================================================
# 2. 准备数据
# ============================================================
plot_df = gene_stats.copy()

# t统计量（用于纵轴）
plot_df["abs_t"] = np.abs(
    plot_df["mean_delta"] /
    (plot_df["sd_delta"] / np.sqrt(plot_df["n_samples"]))
)

plot_df = plot_df.replace([np.inf, -np.inf], np.nan)
plot_df = plot_df.dropna(
    subset=["gene", "mean_delta", "median_delta", "consistency", "fdr", "abs_t"]
).copy()

# strong筛选：决定哪些点着色
strong_sig = (
    (plot_df["fdr"] < 0.05) &
    (plot_df["median_delta"].abs() >= 5) &
    (plot_df["consistency"] >= 0.70)
)

plot_df["group"] = "Other"
plot_df.loc[strong_sig & (plot_df["mean_delta"] < 0), "group"] = "Decrease"
plot_df.loc[strong_sig & (plot_df["mean_delta"] > 0), "group"] = "Increase"

background = plot_df[plot_df["group"] == "Other"].copy()
down = plot_df[plot_df["group"] == "Decrease"].copy()
up = plot_df[plot_df["group"] == "Increase"].copy()

print("全部基因:", len(plot_df))
print("灰色背景基因:", len(background))
print("显著降低年龄基因:", len(down))
print("显著升高年龄基因:", len(up))

# ============================================================
# 3. 自动选择原来左右各6个重点标注基因
#    综合考虑效应大小 + 显著性
# ============================================================
N_LABEL_DOWN = 6
N_LABEL_UP = 6

def pick_label_genes(sub_df, direction="down", n_label=8):
    sub = sub_df.copy()
    if len(sub) == 0:
        return sub

    if direction == "down":
        sub["effect_value"] = -sub["mean_delta"]   # 越大表示降龄效应越强
    else:
        sub["effect_value"] = sub["mean_delta"]    # 越大表示升龄效应越强

    sub["sig_value"] = sub["abs_t"]
    sub["effect_rank"] = sub["effect_value"].rank(method="average", pct=True)
    sub["sig_rank"] = sub["sig_value"].rank(method="average", pct=True)
    sub["label_score"] = sub["effect_rank"] + sub["sig_rank"]

    sub = sub.sort_values(
        ["label_score", "effect_value", "sig_value"],
        ascending=[False, False, False]
    )
    return sub.head(n_label).copy()

label_down_df = pick_label_genes(down, direction="down", n_label=N_LABEL_DOWN)
label_up_df = pick_label_genes(up, direction="up", n_label=N_LABEL_UP)
label_df = pd.concat([label_down_df, label_up_df], axis=0).copy()

print("\n自动标注的降龄基因：")
print(label_down_df[["gene", "mean_delta", "abs_t", "label_score"]])

print("\n自动标注的升龄基因：")
print(label_up_df[["gene", "mean_delta", "abs_t", "label_score"]])

# ============================================================
# 3b. 提取 PeriClock 基因
# ============================================================
PERICLOCK_GENES = ["CST8", "EIF3F", "PNN", "RPS21"]

peri_df = plot_df[plot_df["gene"].isin(PERICLOCK_GENES)].copy()

# 若某个PeriClock基因已经在原始12个标签中，则默认不重复再标一次文字
# 但仍然会用单独颜色高亮散点
already_labeled = set(label_df["gene"])
peri_label_df = peri_df[~peri_df["gene"].isin(already_labeled)].copy()

peri_down_df = peri_label_df[peri_label_df["mean_delta"] < 0].copy()
peri_up_df = peri_label_df[peri_label_df["mean_delta"] > 0].copy()

print("\nPeriClock基因（在plot_df中找到的）:")
if len(peri_df) > 0:
    print(peri_df[["gene", "mean_delta", "abs_t", "fdr", "consistency"]])
else:
    print("一个都没找到")

missing_peri = sorted(set(PERICLOCK_GENES) - set(peri_df["gene"]))
if len(missing_peri) > 0:
    print("以下 PeriClock 基因未在当前数据中找到:", missing_peri)

print("\n需要额外加标签的 PeriClock 基因（排除已在原12个标签中的基因）:")
if len(peri_label_df) > 0:
    print(peri_label_df[["gene", "mean_delta", "abs_t"]])
else:
    print("无（都已经在原标签里）")

# ============================================================
# 4. 颜色设置
# ============================================================
C_GREY = "#BFBFBF"
C_BLUE = "#9CC9E2"       # 浅蓝
C_PINK = "#E7B1B1"       # 肉粉
C_BLUE_DARK = "#3E89B2"
C_PINK_DARK = "#CC747B"

# PeriClock单独高亮颜色
C_PERI = "#F2A541"
C_PERI_DARK = "#D97904"

# ============================================================
# 5. 帮助函数：标签
# ============================================================
def add_labels(ax, data, side, color, x_text, y_start, y_end,
               fontsize=14, fontcolor="#303030", z_line=4, z_text=6):
    if data.empty:
        return

    data = data.sort_values("abs_t", ascending=False).reset_index(drop=True)
    n = len(data)
    label_ys = np.linspace(y_start, y_end, n)

    ha = "right" if side == "left" else "left"

    for (_, row), ty in zip(data.iterrows(), label_ys):
        px, py = row["mean_delta"], row["abs_t"]

        if side == "left":
            elbow = px - 2.3
            line_x = [x_text + 0.9, elbow, elbow, px]
        else:
            elbow = px + 2.3
            line_x = [x_text - 0.9, elbow, elbow, px]

        ax.plot(line_x, [ty, ty, py, py],
                color=color, lw=1.05, alpha=0.92,
                solid_capstyle="round", zorder=z_line)

        ax.text(x_text, ty, row["gene"],
                ha=ha, va="center",
                fontsize=fontsize, fontweight="bold",
                fontstyle="italic", color=fontcolor,
                zorder=z_text)

# ============================================================
# 6. 绘图
# ============================================================
fig, ax = plt.subplots(figsize=(8.2, 5.8))
fig.subplots_adjust(left=0.14, right=0.97, bottom=0.17, top=0.96)

# 所有基因
ax.scatter(background["mean_delta"], background["abs_t"],
           s=16, c=C_GREY, alpha=0.50, edgecolors="none",
           rasterized=True, zorder=1)

ax.scatter(down["mean_delta"], down["abs_t"],
           s=24, c=C_BLUE, alpha=0.78, edgecolors="none",
           rasterized=True, zorder=2)

ax.scatter(up["mean_delta"], up["abs_t"],
           s=24, c=C_PINK, alpha=0.78, edgecolors="none",
           rasterized=True, zorder=2)

# 原来的左右各6个重点基因
ax.scatter(label_down_df["mean_delta"], label_down_df["abs_t"],
           s=88, c=C_BLUE_DARK, edgecolors="white",
           linewidth=1.0, zorder=5)

ax.scatter(label_up_df["mean_delta"], label_up_df["abs_t"],
           s=88, c=C_PINK_DARK, edgecolors="white",
           linewidth=1.0, zorder=5)

# PeriClock基因：单独颜色高亮（覆盖在最上层）
if not peri_df.empty:
    ax.scatter(peri_df["mean_delta"], peri_df["abs_t"],
               s=98, c=C_PERI, edgecolors="white",
               linewidth=1.2, zorder=6)

# ============================================================
# 7. 坐标范围
# ============================================================
ymax = plot_df["abs_t"].max()

ax.set_xlim(-68, 55)
ax.set_ylim(0, ymax + 8)
ax.axvline(0, color="#777777", linestyle="--", linewidth=1.1, zorder=0)

# ============================================================
# 8. 标签排布
#    先画原来的12个，再画PeriClock额外标签
# ============================================================

# 原始左右各6个标签
add_labels(ax, label_down_df, "left", C_BLUE_DARK,
           x_text=-52, y_start=min(ymax + 1, 112), y_end=24,
           fontsize=14, fontcolor="#303030", z_line=4, z_text=6)

add_labels(ax, label_up_df, "right", C_PINK_DARK,
           x_text=43, y_start=min(ymax + 1, 114), y_end=56,
           fontsize=14, fontcolor="#303030", z_line=4, z_text=6)

# 额外的PeriClock标签
# 这些位置你可以根据出图效果再微调
if len(peri_down_df) > 0:
    add_labels(ax, peri_down_df, "left", C_PERI_DARK,
               x_text=-49, y_start=18, y_end=8,
               fontsize=13.5, fontcolor=C_PERI_DARK, z_line=5, z_text=7)

if len(peri_up_df) > 0:
    add_labels(ax, peri_up_df, "right", C_PERI_DARK,
               x_text=50, y_start=50, y_end=38,
               fontsize=13.5, fontcolor=C_PERI_DARK, z_line=5, z_text=7)

# ============================================================
# 9. 图例（可选但建议保留）
# ============================================================
legend_elements = [
    Line2D([0], [0], marker='o', color='w', label='Other',
           markerfacecolor=C_GREY, markeredgecolor='none', markersize=6, alpha=0.8),
    Line2D([0], [0], marker='o', color='w', label='Decrease',
           markerfacecolor=C_BLUE, markeredgecolor='none', markersize=6, alpha=0.9),
    Line2D([0], [0], marker='o', color='w', label='Increase',
           markerfacecolor=C_PINK, markeredgecolor='none', markersize=6, alpha=0.9),
    Line2D([0], [0], marker='o', color='w', label='PeriClock genes',
           markerfacecolor=C_PERI, markeredgecolor='white', markeredgewidth=1.0, markersize=7.5),
]

ax.legend(handles=legend_elements, frameon=False, fontsize=11.5,
          loc='upper center', bbox_to_anchor=(0.52, 1.01), ncol=4,
          handletextpad=0.5, columnspacing=1.2)

# ============================================================
# 10. 坐标轴及文字
# ============================================================
ax.set_xlabel("Mean ΔAge across donors (yr)",
              fontsize=19, fontweight="bold", labelpad=9)

ax.set_ylabel(r"$|t|$ (KO-induced age change)",
              fontsize=19, fontweight="bold", labelpad=10)

ax.tick_params(axis="both", direction="out",
               length=6, width=1.2, labelsize=16)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_linewidth(1.3)
ax.spines["bottom"].set_linewidth(1.3)

# 所有散点保持矢量格式
for collection in ax.collections:
    collection.set_rasterized(False)

plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42

# ============================================================
# 11. 保存
# ============================================================
plt.savefig(f"{outdir}/KO_volcano_with_Periclock.pdf",
            format="pdf", bbox_inches="tight", pad_inches=0.06)

plt.savefig(f"{outdir}/KO_volcano_with_Periclock.png",
            dpi=400, bbox_inches="tight", pad_inches=0.06)

plt.show()

## Pathway enrichment and RNA-processing effects (Fig. 6c–d; Extended Data Fig. 9a–b)

Source: cells 12, 14, 16–18, 32 and 34. All reread results point to the same output tree generated above. The original background, hypergeometric tests, BH correction and pathway/gene filters remain unchanged. GSEApy may download GO BP 2023 / Reactome 2022 GMT libraries if they are not present in the new enrichment output directory. No network enrichment was executed during preparation. Feature-contribution panels (Extended Data Fig. 9c–d) require additional upstream expression-delta files and are outside this tutorial.

In [ ]:
import os
import numpy as np
import pandas as pd
import gseapy as gp
from scipy.stats import hypergeom
from statsmodels.stats.multitest import multipletests

# ============================================================
# 1. 路径和参数
# ============================================================
outdir_enrich = str(OUTPUT_DIR / "results/KO_enrichment")
os.makedirs(outdir_enrich, exist_ok=True)

LIBRARIES = {
    "GO_BP": "GO_Biological_Process_2023",
    "Reactome": "Reactome_2022"
}

MIN_PATHWAY_SIZE = 3
MAX_PATHWAY_SIZE = 300
MIN_OVERLAP = 3
FDR_CUTOFF = 0.05

def clean_genes(x):
    return set(x.dropna().astype(str).str.strip().str.upper()) - {""}

# ============================================================
# 2. 背景基因与两组候选基因
# ============================================================
background = clean_genes(gene_stats.loc[gene_stats["pvalue"].notna(), "gene"])

down_genes = clean_genes(down["gene"]) & background
up_genes = clean_genes(up["gene"]) & background

gene_groups = {
    "Decrease": down_genes,
    "Increase": up_genes
}

print("========== KO基因集合 ==========")
print(f"背景基因: {len(background)}")
print(f"预测年龄降低组: {len(down_genes)}")
print(f"预测年龄升高组: {len(up_genes)}")
print(f"两组重叠基因: {len(down_genes & up_genes)}")

pd.Series(sorted(background), name="gene").to_csv(
    f"{outdir_enrich}/background_genes.csv", index=False
)

for group, genes in gene_groups.items():
    pd.Series(sorted(genes), name="gene").to_csv(
        f"{outdir_enrich}/{group}_genes.csv", index=False
    )

# ============================================================
# 3. 下载GO-BP和Reactome基因集
#    下载过的GMT文件可重复使用
# ============================================================
gene_sets = {}

for db, lib in LIBRARIES.items():
    gmt_file = str(GMT_DIR / f"{lib}.gmt") if (GMT_DIR / f"{lib}.gmt").is_file() else f"{outdir_enrich}/{lib}.gmt"

    if os.path.isfile(gmt_file):
        gs = gp.get_library(name=gmt_file, min_size=0, max_size=100000)
    else:
        gs = gp.get_library(
            name=lib, organism="Human",
            min_size=0, max_size=100000,
            save=gmt_file
        )

    # 只保留在KO背景中存在的基因
    gs = {
        term: set(map(lambda x: str(x).strip().upper(), genes)) & background
        for term, genes in gs.items()
    }

    # 根据与背景的交集大小过滤通路
    gs = {
        term: genes for term, genes in gs.items()
        if MIN_PATHWAY_SIZE <= len(genes) <= MAX_PATHWAY_SIZE
    }

    gene_sets[db] = gs
    print(f"{db}: 保留 {len(gs)} 个通路")

# ============================================================
# 4. ORA：单侧超几何检验
# ============================================================
def run_ora(query_genes, universe, pathways, database, group):
    query = query_genes & universe
    M = len(universe)
    n = len(query)

    if n == 0:
        raise ValueError(f"{group}基因列表为空，无法进行富集分析。")

    results = []

    for term, term_genes in pathways.items():
        K = len(term_genes)
        overlap_genes = query & term_genes
        k = len(overlap_genes)

        # P(X >= k)，检验是否相对于背景过度富集
        pvalue = hypergeom.sf(k - 1, M, K, n) if k > 0 else 1.0

        results.append({
            "Database": database,
            "Group": group,
            "Term": term,
            "Overlap_count": k,
            "Query_size": n,
            "Pathway_size": K,
            "Background_size": M,
            "Gene_ratio": k / n,
            "Background_ratio": K / M,
            "Fold_enrichment": (k / n) / (K / M),
            "P_value": pvalue,
            "Genes": ";".join(sorted(overlap_genes))
        })

    res = pd.DataFrame(results)

    if res.empty:
        return res

    # 每组、每个数据库分别进行BH校正
    res["FDR"] = multipletests(res["P_value"], method="fdr_bh")[1]
    res["Significant"] = (res["FDR"] < FDR_CUTOFF) & (res["Overlap_count"] >= MIN_OVERLAP)

    return res.sort_values(["FDR", "P_value", "Overlap_count"],
                           ascending=[True, True, False]).reset_index(drop=True)

# ============================================================
# 5. 分别对两组基因进行GO-BP和Reactome富集
# ============================================================
enrich_results = {}

for db, pathways in gene_sets.items():
    for group, genes in gene_groups.items():
        res = run_ora(genes, background, pathways, db, group)
        enrich_results[(db, group)] = res

        sig = res[res["Significant"]].copy()
        res.to_csv(f"{outdir_enrich}/{db}_{group}_all.csv", index=False)
        sig.to_csv(f"{outdir_enrich}/{db}_{group}_significant.csv", index=False)

        print(f"\n========== {db} | {group} ==========")
        print(f"输入基因数: {len(genes)}")
        print(f"检验通路数: {len(res)}")
        print(f"显著富集通路数: {len(sig)}")

        if not sig.empty:
            display(sig[["Term", "Overlap_count", "Pathway_size",
                         "Fold_enrichment", "P_value", "FDR"]].head(15).round(4))

# 合并全部结果
enrich_all = pd.concat(enrich_results.values(), ignore_index=True)
enrich_all.to_csv(f"{outdir_enrich}/KO_enrichment_all_results.csv", index=False)

In [ ]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.font_manager as fm
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable
from matplotlib.lines import Line2D
from matplotlib.ticker import FormatStrFormatter, MaxNLocator
from textwrap import fill

# ============================================================
# 1. Arial字体
# ============================================================
font_dir = str(FONT_DIR)

if os.path.isdir(font_dir):
    for f in os.listdir(font_dir):
        if f.lower().endswith((".ttf", ".otf")):
            fm.fontManager.addfont(os.path.join(font_dir, f))

mpl.rcParams.update({
    "font.family": "Arial",
    "font.size": 14,
    "font.weight": "bold",
    "axes.labelweight": "bold",
    "axes.titleweight": "bold",
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "axes.linewidth": 1.2
})

# ============================================================
# 2. 参数
# ============================================================
result_file = str(OUTPUT_DIR / "results/KO_enrichment/KO_enrichment_all_results.csv")
outdir = str(OUTPUT_DIR / "fig")
os.makedirs(outdir, exist_ok=True)

PLOT_TYPE = "bubble"   # "bubble" 或 "bar"
TOP_N = 10
X_COL = "Fold_enrichment"
FDR_CUTOFF = 0.05

TERM_FONTSIZE = 15
TICK_FONTSIZE = 15
LABEL_FONTSIZE = 17
TITLE_FONTSIZE = 18
CBAR_TICKSIZE = 12
LEGEND_FONTSIZE = 13

# 通路名称自动换行宽度
WRAP_WIDTH = 41

# 配色
CMAPS = {
    "Decrease": "Blues",
    "Increase": "Reds"
}

# 气泡大小
SIZE_BASE = 35
SIZE_SCALE = 19

def bubble_size(n):
    return SIZE_BASE + SIZE_SCALE * np.asarray(n)

# colorbar最小跨度：避免像最后一张图那样范围过窄
MIN_COLOR_SPAN = 0.20


from matplotlib.colors import LinearSegmentedColormap

def truncate_colormap(cmap_name, minval=0.22, maxval=1.0, n=256):
    base = mpl.colormaps[cmap_name]
    new_colors = base(np.linspace(minval, maxval, n))
    return LinearSegmentedColormap.from_list(
        f"trunc_{cmap_name}", new_colors
    )
    

# ============================================================
# 3. 读取并整理富集结果
# ============================================================
enrich_df = pd.read_csv(result_file)

for col in ["FDR", "P_value", "Overlap_count",
            "Fold_enrichment", "Gene_ratio"]:
    enrich_df[col] = pd.to_numeric(enrich_df[col], errors="coerce")

enrich_df["Database"] = enrich_df["Database"].astype(str).str.strip()
enrich_df["Group"] = enrich_df["Group"].astype(str).str.strip()

enrich_df["Significant"] = (
    enrich_df["Significant"].astype(str).str.lower().isin(["true", "1"])
)

def clean_term(term):
    term = re.sub(r"\s*\(\s*GO:\s*\d+\s*\)", "", str(term), flags=re.I)
    term = term.replace("_", " ")
    term = re.sub(r"\s+", " ", term).strip()
    return term

enrich_df["Term_clean"] = enrich_df["Term"].apply(clean_term)
enrich_df["neglog10_fdr"] = -np.log10(enrich_df["FDR"].clip(lower=1e-300))

# ============================================================
# 4. 筛选每组Top N通路
# ============================================================
plot_data = {}

for database in ["GO_BP", "Reactome"]:
    for group in ["Decrease", "Increase"]:
        sub = enrich_df[
            (enrich_df["Database"] == database) &
            (enrich_df["Group"] == group) &
            (enrich_df["Significant"]) &
            (enrich_df["FDR"] < FDR_CUTOFF)
        ].copy()

        sub = sub.sort_values(
            ["FDR", "P_value", "Overlap_count"],
            ascending=[True, True, False]
        ).head(TOP_N)

        plot_data[(database, group)] = sub

all_plot = pd.concat(plot_data.values(), ignore_index=True)

if all_plot.empty:
    raise ValueError("没有符合条件的显著富集通路。")

count_values = sorted(set(
    np.rint(np.quantile(
        all_plot["Overlap_count"], [0.1, 0.5, 0.9]
    )).astype(int)
))

print("统一Overlap count图例:", count_values)

# ============================================================
# 5. 绘图函数
# ============================================================
def plot_classic_enrichment(database, group):
    sub = plot_data[(database, group)].copy()

    if sub.empty:
        print(f"{database} - {group}: 无显著通路")
        return

    sub["Term_display"] = sub["Term_clean"].apply(
        lambda x: fill(x, width=WRAP_WIDTH,
                       break_long_words=False,
                       break_on_hyphens=False)
    )

    # 最显著的放上面
    sub = sub.sort_values(
        ["FDR", "P_value"], ascending=[False, False]
    ).reset_index(drop=True)

    # ========================================================
    # A. 当前图单独颜色范围，但给一个最小跨度
    # ========================================================
    local_min = sub["neglog10_fdr"].min()
    local_max = sub["neglog10_fdr"].max()

    center = (local_min + local_max) / 2
    half_span = max((local_max - local_min) / 2, MIN_COLOR_SPAN / 2)

    color_min = max(0, center - half_span)
    color_max = center + half_span

    norm = Normalize(vmin=color_min, vmax=color_max)

    # ========================================================
    # B. 根据通路名称行数决定图高
    # ========================================================
    n_lines = sub["Term_display"].str.count("\n") + 1
    row_heights = np.maximum(
        0.58 + (n_lines - 1) * 0.32,
        0.63
    ).to_numpy()

    y_positions = np.cumsum(row_heights) - row_heights / 2
    total_height = row_heights.sum()

    fig_height = max(5.0, total_height + 1.45)
    fig, ax = plt.subplots(figsize=(9.4, fig_height))

    # 比原来更紧凑
    fig.subplots_adjust(
        left=0.42, right=0.82,
        bottom=0.13, top=0.90
    )

    cmap = truncate_colormap(CMAPS[group], minval=0.22, maxval=1.0)

    # ========================================================
    # C. 作图
    # ========================================================
    if PLOT_TYPE == "bubble":
        ax.scatter(
            sub[X_COL],
            y_positions,
            s=bubble_size(sub["Overlap_count"]),
            c=sub["neglog10_fdr"],
            cmap=cmap,
            norm=norm,
            edgecolors="white",
            linewidths=0.9,
            alpha=0.92,
            zorder=3
        )

    elif PLOT_TYPE == "bar":
        ax.barh(
            y_positions,
            sub[X_COL],
            height=row_heights * 0.58,
            color=cmap(norm(sub["neglog10_fdr"].to_numpy())),
            edgecolor="none",
            zorder=3
        )

    else:
        raise ValueError("PLOT_TYPE只能是 'bubble' 或 'bar'")

    # ========================================================
    # D. 轴和标签
    # ========================================================
    ax.set_yticks(y_positions)
    ax.set_yticklabels(
        sub["Term_display"],
        fontsize=TERM_FONTSIZE,
        fontweight="bold"
    )

    ax.set_ylim(0, total_height)

    ax.set_xlabel(
        "Fold enrichment" if X_COL == "Fold_enrichment" else "Gene ratio",
        fontsize=LABEL_FONTSIZE,
        fontweight="bold",
        labelpad=8
    )

    # 减少右侧留白
    xmax = sub[X_COL].max()
    ax.set_xlim(0, xmax * 1.08)

    for y in y_positions:
        ax.axhline(y, color="#EAEAEA", lw=0.7, zorder=0)

    ax.tick_params(axis="x", labelsize=TICK_FONTSIZE, length=5, width=1.1)
    ax.tick_params(axis="y", length=0, pad=8)

    for tick in ax.get_xticklabels():
        tick.set_fontweight("bold")

    ax.spines[["top", "right", "left"]].set_visible(False)
    ax.spines["bottom"].set_linewidth(1.2)

    group_title = "Age decrease" if group == "Decrease" else "Age increase"
    ax.set_title(
        f"{database} | {group_title}",
        fontsize=TITLE_FONTSIZE,
        fontweight="bold",
        pad=12
    )

    # ========================================================
    # E. colorbar：更靠近主图，且保留两位小数
    # ========================================================
    cax = fig.add_axes([0.84, 0.42, 0.026, 0.34])

    sm = ScalarMappable(norm=norm, cmap=cmap)
    sm.set_array([])

    cbar = fig.colorbar(sm, cax=cax)
    cbar.set_label(
        r"$-\log_{10}(\mathrm{FDR})$",
        fontsize=14, fontweight="bold", labelpad=8
    )
    cbar.ax.tick_params(labelsize=CBAR_TICKSIZE, width=1)

    # 两位小数
    cbar.ax.yaxis.set_major_formatter(FormatStrFormatter('%.2f'))

    # 控制colorbar刻度数量，不要过密
    cbar.locator = MaxNLocator(nbins=4)
    cbar.update_ticks()

    for tick in cbar.ax.get_yticklabels():
        tick.set_fontweight("bold")

    # ========================================================
    # F. 气泡大小图例：更靠近主图
    # ========================================================
    if PLOT_TYPE == "bubble":
        handles = [
            Line2D(
                [], [],
                marker="o", linestyle="None",
                markersize=np.sqrt(float(bubble_size(v))),
                markerfacecolor="#A6A6A6",
                markeredgecolor="white",
                label=str(v)
            )
            for v in count_values
        ]

        legend = ax.legend(
            handles=handles,
            title="Overlap count",
            bbox_to_anchor=(1.02, 0.30),
            loc="upper left",
            frameon=False,
            fontsize=LEGEND_FONTSIZE,
            title_fontsize=LEGEND_FONTSIZE + 1,
            labelspacing=1.1,
            borderaxespad=0
        )
        legend.get_title().set_fontweight("bold")

        for txt in legend.get_texts():
            txt.set_fontweight("bold")

    # ========================================================
    # G. 保存
    # ========================================================
    name = f"{database}_{group}_classic_{PLOT_TYPE}"

    fig.savefig(
        f"{outdir}/{name}.pdf",
        bbox_inches="tight",
        pad_inches=0.06
    )
    fig.savefig(
        f"{outdir}/{name}.png",
        dpi=400,
        bbox_inches="tight",
        pad_inches=0.06
    )

    plt.show()
    plt.close(fig)

# ============================================================
# 6. 绘制四张图
# ============================================================
for database in ["GO_BP", "Reactome"]:
    for group in ["Decrease", "Increase"]:
        plot_classic_enrichment(database, group)

In [ ]:
# 提取每条通路命中的 KO 基因（Top10通路）
import os
import re
import pandas as pd
import numpy as np

# ============================================================
# 1. 读取富集结果
# ============================================================
result_dir = str(OUTPUT_DIR / "results/KO_enrichment")
enrich_df = pd.read_csv(f"{result_dir}/KO_enrichment_all_results.csv")

TOP_N = 10

enrich_df["Significant"] = enrich_df["Significant"].astype(str).str.lower().isin(["true", "1"])

def clean_term(term):
    term = re.sub(r"\s*\(\s*GO:\s*\d+\s*\)", "", str(term), flags=re.I)
    term = re.sub(r"\s+", " ", term.replace("_", " ")).strip()
    term = re.sub(r"\s+R-HSA-\d+$", "", term)
    return term

enrich_df["Term_clean"] = enrich_df["Term"].apply(clean_term)

# ============================================================
# 2. 提取四组Top 10显著通路
# ============================================================
top_pathways = []

for db in ["GO_BP", "Reactome"]:
    for group in ["Decrease", "Increase"]:
        sub = enrich_df[
            (enrich_df["Database"] == db) &
            (enrich_df["Group"] == group) &
            (enrich_df["Significant"])
        ].copy()

        sub = sub.sort_values(["FDR", "P_value"]).head(TOP_N)
        top_pathways.append(sub)

top_pathways = pd.concat(top_pathways, ignore_index=True)

# ============================================================
# 3. 展示每条通路及命中KO基因
# ============================================================
for db in ["GO_BP", "Reactome"]:
    for group in ["Decrease", "Increase"]:
        sub = top_pathways[
            (top_pathways["Database"] == db) &
            (top_pathways["Group"] == group)
        ]

        print(f"\n========== {db} | {group} ==========")
        display(sub[[
            "Term_clean", "Overlap_count", "Fold_enrichment", "FDR", "Genes"
        ]].reset_index(drop=True))

# 保存
top_pathways.to_csv(f"{result_dir}/Top_pathways_KO_genes.csv", index=False)

In [ ]:
# 基因的KO效应
# ============================================================
# 4. 将通路基因拆分成长表
# ============================================================
pathway_genes = top_pathways[
    ["Database", "Group", "Term_clean", "Genes"]
].copy()

pathway_genes["gene"] = pathway_genes["Genes"].fillna("").str.split(";")
pathway_genes = pathway_genes.explode("gene")
pathway_genes["gene"] = pathway_genes["gene"].str.strip().str.upper()
pathway_genes = pathway_genes[pathway_genes["gene"] != ""].drop(columns="Genes")

# ============================================================
# 5. 合并每个基因的KO年龄变化
# ============================================================
gene_effect = gene_stats[[
    "gene", "mean_delta", "median_delta",
    "consistency", "frac_decrease", "frac_increase", "fdr"
]].copy()

gene_effect["gene"] = gene_effect["gene"].str.upper()

pathway_gene_effects = pathway_genes.merge(
    gene_effect, on="gene", how="left", validate="many_to_one"
)

# ============================================================
# 6. 查看每条通路的KO基因及其效应
# ============================================================
for db in ["GO_BP", "Reactome"]:
    for group in ["Decrease", "Increase"]:
        sub = pathway_gene_effects[
            (pathway_gene_effects["Database"] == db) &
            (pathway_gene_effects["Group"] == group)
        ]

        print(f"\n========== {db} | {group} ==========")

        display(
            sub[[
                "Term_clean", "gene", "mean_delta",
                "median_delta", "consistency", "fdr"
            ]].sort_values(
                ["Term_clean", "mean_delta"]
            ).reset_index(drop=True).round(4)
        )

pathway_gene_effects.to_csv(
    f"{result_dir}/Top_pathways_gene_KO_effects.csv", index=False
)

In [ ]:
# 参与多条通路富集的基因
# ============================================================
# 7. 基因参与通路数量
# ============================================================
gene_pathway_summary = (
    pathway_gene_effects
    .groupby(["Group", "gene"])
    .agg(
        n_pathways=("Term_clean", "nunique"),
        pathways=("Term_clean", lambda x: "; ".join(sorted(set(x)))),
        mean_delta=("mean_delta", "first"),
        median_delta=("median_delta", "first"),
        consistency=("consistency", "first")
    )
    .reset_index()
)

# 按涉及通路数量和KO效应大小排序
gene_pathway_summary["abs_effect"] = gene_pathway_summary["mean_delta"].abs()

gene_pathway_summary = gene_pathway_summary.sort_values(
    ["Group", "n_pathways", "abs_effect"],
    ascending=[True, False, False]
)

for group in ["Decrease", "Increase"]:
    print(f"\n========== {group}：多通路关联基因 ==========")
    display(
        gene_pathway_summary[
            gene_pathway_summary["Group"] == group
        ].head(30).round(4)
    )

gene_pathway_summary.to_csv(
    f"{result_dir}/KO_gene_pathway_summary.csv", index=False
)

In [ ]:
import os
import numpy as np
import pandas as pd

# ============================================================
# 1. 路径与读取数据
# ============================================================
base = str(OUTPUT_DIR)
pathway_file = f"{base}/results/KO_enrichment/Top_pathways_gene_KO_effects.csv"
outdir = f"{base}/results/KO_same_module_bidirectional"
os.makedirs(outdir, exist_ok=True)

df = pd.read_csv(pathway_file)
df["gene"] = df["gene"].astype(str).str.strip().str.upper()
df["Term_clean"] = df["Term_clean"].astype(str).str.strip()

# ============================================================
# 2. 提取Reactome中rRNA processing相关通路
# ============================================================
module_df = df[
    (df["Database"] == "Reactome") &
    (df["Term_clean"].str.contains("rRNA processing", case=False, na=False))
].copy()

print("========== 涉及的Reactome通路 ==========")
display(module_df[["Group", "Term_clean"]].drop_duplicates().reset_index(drop=True))

# ============================================================
# 3. 提取双向KO基因并去重
# ============================================================
module_gene_stats = module_df.drop_duplicates(["Group", "gene"]).copy()
module_gene_stats["KO_direction"] = module_gene_stats["Group"].map({
    "Decrease": "Age decrease",
    "Increase": "Age increase"
})

module_gene_stats = pd.concat([
    module_gene_stats[module_gene_stats["Group"] == "Decrease"].sort_values("median_delta"),
    module_gene_stats[module_gene_stats["Group"] == "Increase"].sort_values("median_delta", ascending=False)
], ignore_index=True)

# ============================================================
# 4. 展示和保存
# ============================================================
for group in ["Age decrease", "Age increase"]:
    sub = module_gene_stats[module_gene_stats["KO_direction"] == group]

    print(f"\n========== {group}：{len(sub)} genes ==========")
    display(sub[[
        "gene", "mean_delta", "median_delta", "consistency", "fdr"
    ]].reset_index(drop=True).round(4))

module_gene_stats.to_csv(
    f"{outdir}/rRNA_processing_bidirectional_genes.csv",
    index=False
)

print(f"\n结果保存至：{outdir}/rRNA_processing_bidirectional_genes.csv")

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from matplotlib.patches import Patch
from matplotlib.ticker import MultipleLocator

# ============================================================
# 1. 路径与字体
# ============================================================
base = str(OUTPUT_DIR)
infile = f"{base}/results/KO_same_module_bidirectional/rRNA_processing_bidirectional_genes.csv"
figdir = f"{base}/fig/KO_same_module_bidirectional"
os.makedirs(figdir, exist_ok=True)

font_dir = str(FONT_DIR)
if not os.path.isdir(font_dir):
    font_dir = str(FONT_DIR)

if os.path.isdir(font_dir):
    for f in os.listdir(font_dir):
        if f.lower().endswith((".ttf", ".otf")):
            fm.fontManager.addfont(os.path.join(font_dir, f))

mpl.rcParams.update({
    "font.family": "Arial",
    "font.sans-serif": ["Arial"],
    "font.size": 12,
    "font.weight": "bold",
    "axes.labelsize": 14,
    "axes.labelweight": "bold",
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "axes.linewidth": 1.1,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "savefig.facecolor": "white"
})

# ============================================================
# 2. 绘图参数
# ============================================================
TOP_DOWN = 8
TOP_UP = 8

C_BLUE = "#82B6D1"
C_RED = "#DEA0A8"
C_TEXT = "#303030"
C_ZERO = "#777777"

# ============================================================
# 3. 读取并筛选基因
# ============================================================
df = pd.read_csv(infile)
df["gene"] = df["gene"].astype(str).str.strip().str.upper()
df["Group"] = df["Group"].astype(str).str.strip()
df["median_delta"] = pd.to_numeric(df["median_delta"], errors="coerce")
df = df.dropna(subset=["median_delta"]).drop_duplicates(["Group", "gene"]).copy()

# 防止不同方向的基因被错误混入
assert (df.loc[df["Group"] == "Decrease", "median_delta"] < 0).all()
assert (df.loc[df["Group"] == "Increase", "median_delta"] > 0).all()

down_all = df[df["Group"] == "Decrease"].sort_values("median_delta")
up_all = df[df["Group"] == "Increase"].sort_values("median_delta", ascending=False)

down = down_all.head(TOP_DOWN).copy()
up = up_all.head(TOP_UP).copy()

plot_df = pd.concat([down, up], ignore_index=True)

n_down, n_up = len(down), len(up)

print("========== rRNA processing ==========")
print(f"全部降龄基因: {len(down_all)}，展示: {n_down}")
print(f"全部升龄基因: {len(up_all)}，展示: {n_up}")

display(
    plot_df[["gene", "Group", "median_delta", "consistency", "fdr"]]
    .reset_index(drop=True).round(4)
)

plot_df.to_csv(
    f"{figdir}/rRNA_processing_main_figure_genes.csv",
    index=False
)

# ============================================================
# 4. 设置纵轴位置
# ============================================================
# 让两个方向之间存在少量间隔
gap = 0.65
y_down = np.arange(n_down, dtype=float)
y_up = np.arange(n_up, dtype=float) + n_down + gap

ypos = np.concatenate([y_down, y_up])
plot_df["y"] = ypos

# ============================================================
# 5. 绘制主图
# ============================================================
fig, ax = plt.subplots(figsize=(7.0, 6.2))

# 颜色与柱子
bar_colors = np.where(
    plot_df["median_delta"] < 0, C_BLUE, C_RED
)

ax.barh(
    plot_df["y"],
    plot_df["median_delta"],
    height=0.74,
    color=bar_colors,
    edgecolor="none",
    zorder=3
)

# 0参考线
ax.axvline(
    0, color=C_ZERO,
    linestyle=(0, (3, 3)),
    linewidth=1.0, zorder=1
)

# 两组之间的轻微分隔
if n_down and n_up:
    split_y = (y_down[-1] + y_up[0]) / 2
    ax.axhline(
        split_y, color="#D6D6D6",
        linewidth=0.85, zorder=1
    )

# ============================================================
# 6. 基因名和数值标签
# ============================================================
ax.set_yticks(plot_df["y"])
ax.set_yticklabels(plot_df["gene"], fontsize=11.5, fontweight="bold")

# 按最大效应自动设置x范围，为数值标签留足空间
xmin_raw = plot_df["median_delta"].min()
xmax_raw = plot_df["median_delta"].max()

xmin = np.floor((xmin_raw - 9) / 10) * 10
xmax = np.ceil((xmax_raw + 9) / 10) * 10
ax.set_xlim(xmin, xmax)

# 数值标注
for _, row in plot_df.iterrows():
    x, y = row["median_delta"], row["y"]

    ax.text(
        x - 1.1 if x < 0 else x + 1.1,
        y,
        f"{x:+.2f}",
        ha="right" if x < 0 else "left",
        va="center",
        fontsize=10.2,
        fontweight="bold",
        color=C_TEXT
    )

# 按列表顺序自上而下展示
ax.set_ylim(plot_df["y"].max() + 0.65, -0.65)

# ============================================================
# 7. 坐标轴美化
# ============================================================
ax.set_xlabel(
    "Median ΔAge (yr)",
    fontsize=14.5,
    fontweight="bold",
    labelpad=7
)

ax.xaxis.set_major_locator(MultipleLocator(10))
ax.tick_params(
    axis="x", direction="out",
    length=4.5, width=1.1,
    labelsize=11
)
ax.tick_params(axis="y", length=0, pad=7)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_visible(False)
ax.spines["bottom"].set_linewidth(1.1)

# ============================================================
# 8. 标题与图例：单独排版，避免重叠
# ============================================================
fig.text(
    0.54, 0.965,
    "rRNA processing",
    ha="center", va="top",
    fontsize=16,
    fontweight="bold"
)

legend_elements = [
    Patch(facecolor=C_BLUE, label=f"Age decrease (n={n_down})"),
    Patch(facecolor=C_RED, label=f"Age increase (n={n_up})")
]

fig.legend(
    handles=legend_elements,
    loc="upper center",
    bbox_to_anchor=(0.54, 0.925),
    ncol=2,
    frameon=False,
    fontsize=11,
    handlelength=1.5,
    columnspacing=1.8
)

# ============================================================
# 9. 调整留白
# ============================================================
fig.subplots_adjust(
    left=0.145,
    right=0.965,
    top=0.835,
    bottom=0.105
)

# ============================================================
# 10. 保存
# ============================================================
output = f"{figdir}/rRNA_processing_bidirectional_KO_main"

fig.savefig(
    f"{output}.pdf",
    bbox_inches="tight",
    pad_inches=0.05
)

fig.savefig(
    f"{output}.png",
    dpi=400,
    bbox_inches="tight",
    pad_inches=0.05
)

plt.show()
plt.close(fig)

print(f"\nPDF: {output}.pdf")
print(f"PNG: {output}.png")